# Preprocesamiento — Punto 2: Regresión para Consultoría en Desarrollo Global
**Taller 1 · Consultoría Económica con IA** | David Rodríguez · Juan Rueda · 2026

In [46]:
%matplotlib inline
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

NOTEBOOK_DIR = Path().resolve()
DIR_DATOS    = NOTEBOOK_DIR / "Datos"
DIR_VIZS     = NOTEBOOK_DIR / "Visualizaciones"
LOCAL_TRAIN  = DIR_DATOS / "Life Expectancy Data.csv"
URL_TRAIN    = (
    "https://raw.githubusercontent.com/darc-17/Sandbox_HE2_"
    "DavidRodriguez/refs/heads/main/Taller%201/Punto%202.%20"
    "Regresi%C3%B3n%20para%20Consultor%C3%ADa%20en%20Desarrollo%20Global/Life%20Expectancy%20Data.csv"
)

# 1. Carga de Datos y Nombres Variables

In [47]:
if LOCAL_TRAIN.exists():
    df = pd.read_csv(LOCAL_TRAIN)
    print("Fuente: archivo local")
else:
    df = pd.read_csv(URL_TRAIN)
    print("Fuente: URL remota (archivo local no encontrado)")

print(f"Forma inicial: {df.shape[0]:,} obs · {df.shape[1]} variables")
print(f"Tipos de dato:\n{df.dtypes.value_counts().rename('columnas').to_string()}")

Fuente: URL remota (archivo local no encontrado)
Forma inicial: 2,938 obs · 22 variables
Tipos de dato:
float64    16
int64       4
str         2


In [48]:
# ── Normalizar y renombrar columnas ───────────────────────────────────────────
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

rename_map = {
    'Status':                          'develp_status',
    'Life expectancy':                 'life_expectancy',
    'Adult Mortality':                 'adult_mortality',
    'infant deaths':                   'infant_deaths',
    'Alcohol':                         'alcohol',
    'percentage expenditure':          'health_exp_pct_gdp',
    'Hepatitis B':                     'hepatitis_b',
    'Measles':                         'measles',
    'BMI':                             'bmi',
    'under-five deaths':               'under5_deaths',
    'Polio':                           'polio',
    'Total expenditure':               'gov_health_exp_pct',
    'Diphtheria':                      'diphtheria',
    'HIV/AIDS':                        'hiv_aids',
    'GDP':                             'gdp_per_capita',
    'Population':                      'population',
    'thinness 1-19 years':             'thinness_10_19',
    'thinness 5-9 years':              'thinness_5_9',
    'Income composition of resources': 'hdi_income',
    'Schooling':                       'schooling',
}

df = df.rename(columns=rename_map)
print(f"Columnas renombradas. Total: {df.shape[1]}")
print(list(df.columns))


Columnas renombradas. Total: 22
['Country', 'Year', 'develp_status', 'life_expectancy', 'adult_mortality', 'infant_deaths', 'alcohol', 'health_exp_pct_gdp', 'hepatitis_b', 'measles', 'bmi', 'under5_deaths', 'polio', 'gov_health_exp_pct', 'diphtheria', 'hiv_aids', 'gdp_per_capita', 'population', 'thinness_10_19', 'thinness_5_9', 'hdi_income', 'schooling']


# 2. Revisamos Duplicados y Missings

In [49]:
# ── Duplicados ────────────────────────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f"Filas duplicadas: {n_dup} ({n_dup/len(df)*100:.2f} %)")
if n_dup:
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)))

# ── Missings ──────────────────────────────────────────────────────────────────
miss = (
    pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean() * 100,
    })
    .query("n_missing > 0")
    .sort_values("pct_missing", ascending=False)
)
print(f"\nVariables con valores faltantes: {len(miss)} de {df.shape[1]}")

# ── Columnas con missings ──────────────────────────────────
if not miss.empty:
    print("\nColumnas con missings (n):")
    for col, row in miss.iterrows():
        print(f"  {col:<40} {int(row['n_missing']):>5} ({row['pct_missing']:.2f}%)")


Filas duplicadas: 0 (0.00 %)

Variables con valores faltantes: 14 de 22

Columnas con missings (n):
  population                                 652 (22.19%)
  hepatitis_b                                553 (18.82%)
  gdp_per_capita                             448 (15.25%)
  gov_health_exp_pct                         226 (7.69%)
  alcohol                                    194 (6.60%)
  hdi_income                                 167 (5.68%)
  schooling                                  163 (5.55%)
  thinness_5_9                                34 (1.16%)
  thinness_10_19                              34 (1.16%)
  bmi                                         34 (1.16%)
  polio                                       19 (0.65%)
  diphtheria                                  19 (0.65%)
  life_expectancy                             10 (0.34%)
  adult_mortality                             10 (0.34%)


## 2.1 Países con obs solo en 2013 eliminados

In [50]:
# Explorando con Data Wrangler podemos ver que hay 193 países y los años son
# del 2001 al 2015 (16 años) con 10 más obs en 2013 (países que seguramente solo
# están ese año)

todos_anios = set(df["Year"].unique())
paises_2013 = set(df[df["Year"] == 2013]["Country"].unique())

paises_incompletos = sorted([
    p for p in paises_2013
    if set(df[df["Country"] == p]["Year"].unique()) != todos_anios
])

print(f"Países en 2013 con años faltantes: {len(paises_incompletos)}")
for p in paises_incompletos:
    anios_presentes = sorted(df[df["Country"] == p]["Year"].unique())
    anios_faltantes = sorted(todos_anios - set(anios_presentes))
    print(f"  {p}: faltan {anios_faltantes}")

df = df[~df["Country"].isin(paises_incompletos)].reset_index(drop=True)
print(f"Shape tras eliminar países incompletos: {df.shape}")

Países en 2013 con años faltantes: 10
  Cook Islands: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015)]
  Dominica: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015)]
  Marshall Islands: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015)]
  Monaco: faltan [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64

## 2.2 South Sudan eliminado por muchos missings

In [51]:
df = df[df["Country"] != "South Sudan"].reset_index(drop=True)
print(f"Shape tras eliminar South Sudan: {df.shape}")
df

Shape tras eliminar South Sudan: (2912, 22)


,Country,Year,develp_status,life_expectancy,adult_mortality,infant_deaths,alcohol,health_exp_pct_gdp,hepatitis_b,measles,bmi,under5_deaths,polio,gov_health_exp_pct,diphtheria,hiv_aids,gdp_per_capita,population,thinness_10_19,thinness_5_9,hdi_income,schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,19.1,83,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,18.6,86,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,18.1,89,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,17.6,93,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,17.2,97,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2907,Zimbabwe,2004,Developing,44.3,723.0,27,4.36,0.000000,68.0,31,27.1,42,67.0,7.13,65.0,33.6,454.366654,12777511.0,9.4,9.4,0.407,9.2
2908,Zimbabwe,2003,Developing,44.5,715.0,26,4.06,0.000000,7.0,998,26.7,41,7.0,6.52,68.0,36.7,453.351155,12633897.0,9.8,9.9,0.418,9.5
2909,Zimbabwe,2002,Developing,44.8,73.0,25,4.43,0.000000,73.0,304,26.3,40,73.0,6.53,71.0,39.8,57.348340,125525.0,1.2,1.3,0.427,10.0
2910,Zimbabwe,2001,Developing,45.3,686.0,25,1.72,0.000000,76.0,529,25.9,39,76.0,6.16,75.0,42.1,548.587312,12366165.0,1.6,1.7,0.427,9.8
